# Korean Olympiad dataset inspection

This Colab notebook inspects two datasets without modifying them:

1. `ChuGyouk/AI-MO-NuminaMath-CoT-Ko`: count every `source`, keep only rows whose source is exactly `olympiads`, and print examples.
2. `ChuGyouk/OlympiadBench-Math-Ko`: print its schema, row count, and examples.

NuminaMath is streamed, and its first pass reads only the `source` column. This avoids loading all 859k long solutions into system RAM. A complete text report is saved for easy sharing.

In [ ]:
%pip install -q -U datasets huggingface_hub tqdm

In [ ]:
from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

print("Using HF_TOKEN from Colab Secrets" if HF_TOKEN else "HF_TOKEN not found; using public access")

In [ ]:
# ---- User controls ----
NUMINA_DATASET_ID = "ChuGyouk/AI-MO-NuminaMath-CoT-Ko"
NUMINA_SPLIT = "train"
TARGET_SOURCE = "olympiads"  # Exact, case-sensitive match.

OLYMPIAD_BENCH_DATASET_ID = "ChuGyouk/OlympiadBench-Math-Ko"
OLYMPIAD_BENCH_SPLIT = "test"

EXAMPLE_ROWS = 3
MAX_TEXT_CHARACTERS = 6000  # Display limit per long string; does not alter dataset rows.
REPORT_PATH = "/content/korean_olympiad_dataset_report.txt"

In [ ]:
import json
from collections import Counter
from pathlib import Path

from datasets import load_dataset
from tqdm.auto import tqdm


report_lines = []


def emit(value=""):
    text = str(value)
    report_lines.append(text)
    print(text)


def limit_long_strings(value):
    if isinstance(value, str) and len(value) > MAX_TEXT_CHARACTERS:
        removed = len(value) - MAX_TEXT_CHARACTERS
        return value[:MAX_TEXT_CHARACTERS] + f"\n... [display truncated by {removed:,} characters]"
    if isinstance(value, list):
        return [limit_long_strings(item) for item in value]
    if isinstance(value, dict):
        return {key: limit_long_strings(item) for key, item in value.items()}
    return value


def emit_json(value):
    emit(json.dumps(limit_long_strings(value), ensure_ascii=False, indent=2, default=str))


def emit_examples(rows):
    for index, row in enumerate(rows):
        emit(f"--- EXAMPLE {index + 1} ---")
        emit_json(row)

## 1. Count NuminaMath sources

This cell streams only the small `source` column across the dataset. It prints the distribution of every source and the exact number matching `olympiads`.

In [ ]:
emit("=" * 100)
emit(f"DATASET: {NUMINA_DATASET_ID}")
emit(f"SPLIT: {NUMINA_SPLIT}")
emit("Counting source values with streaming...")

numina_source_only = load_dataset(
    NUMINA_DATASET_ID,
    split=NUMINA_SPLIT,
    streaming=True,
    token=HF_TOKEN,
).select_columns(["source"])

source_counts = Counter()
for row in tqdm(numina_source_only, desc="Scanning NuminaMath source column"):
    source_counts[row["source"]] += 1

total_numina_rows = sum(source_counts.values())
target_numina_rows = source_counts.get(TARGET_SOURCE, 0)

emit(f"TOTAL ROWS: {total_numina_rows:,}")
emit("SOURCE COUNTS:")
for source, count in source_counts.most_common():
    emit(f"  {source!r}: {count:,} ({count / total_numina_rows:.2%})")
emit()
emit(f"EXACT FILTER: source == {TARGET_SOURCE!r}")
emit(f"MATCHING ROWS: {target_numina_rows:,}")

if target_numina_rows == 0:
    raise ValueError(
        f"No rows matched source == {TARGET_SOURCE!r}. "
        f"Available values: {list(source_counts)}"
    )

## 2. Print filtered NuminaMath examples

In [ ]:
numina_stream = load_dataset(
    NUMINA_DATASET_ID,
    split=NUMINA_SPLIT,
    streaming=True,
    token=HF_TOKEN,
)
numina_olympiads = numina_stream.filter(lambda row: row["source"] == TARGET_SOURCE)
numina_examples = list(numina_olympiads.take(EXAMPLE_ROWS))

emit()
emit("NUMINAMATH FILTERED SCHEMA:")
emit(f"COLUMNS: {list(numina_stream.features) if numina_stream.features else list(numina_examples[0])}")
if numina_stream.features:
    emit("FEATURES:")
    emit_json(numina_stream.features.to_dict())
emit(f"FIRST {len(numina_examples)} ROWS MATCHING source == {TARGET_SOURCE!r}:")
emit_examples(numina_examples)

## 3. Inspect OlympiadBench-Math-Ko

In [ ]:
olympiad_bench = load_dataset(
    OLYMPIAD_BENCH_DATASET_ID,
    split=OLYMPIAD_BENCH_SPLIT,
    token=HF_TOKEN,
)

emit()
emit("=" * 100)
emit(f"DATASET: {OLYMPIAD_BENCH_DATASET_ID}")
emit(f"SPLIT: {OLYMPIAD_BENCH_SPLIT}")
emit(f"TOTAL ROWS: {len(olympiad_bench):,}")
emit(f"COLUMNS: {olympiad_bench.column_names}")
emit("FEATURES:")
emit_json(olympiad_bench.features.to_dict())
emit(f"FIRST {min(EXAMPLE_ROWS, len(olympiad_bench))} ROWS:")
emit_examples([olympiad_bench[index] for index in range(min(EXAMPLE_ROWS, len(olympiad_bench)))])

In [ ]:
report_text = "\n".join(report_lines)
Path(REPORT_PATH).write_text(report_text, encoding="utf-8")
print(f"\nSaved complete report to {REPORT_PATH}")

In [ ]:
# Download this report and attach it to Codex.
from google.colab import files

files.download(REPORT_PATH)